In [1]:
from pathlib import Path
import sys
CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

secrets = CWD / "secrets.env"
if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

True

In [2]:
class MINER(object):
	async def ingest(self, preprocess_results_filename: str):
		""" Ingest knowledge from some text. """
		pass
	async def pre_retrieve(self):
		pass
	async def retrieve(self, text: str) -> str:
		""" Find information relevant to a text. """
		pass
	async def reset(self):
		""" Forget ingested knowledge. """
		pass

In [3]:
target_dataset_file:Path = CWD / "../datasets/MINE/A Brief History of Time Zones.json"
if target_dataset_file.is_file():
    print(f"found {target_dataset_file}.. ")

# await process_dataset_file()

found /home/nathan/Projects/X-RAG/src/../datasets/MINE/A Brief History of Time Zones.json.. 


In [4]:
import utils.mg_driver as mg_driver
from upsert import upsert_from_preprocessed
from leanrag_build import build
import leanrag_retrieve

await mg_driver.init()

class LeanragMINER(MINER):

    async def ingest(self, preprocess_chunks_filepath:Path, preprocess_descriptions_filepath:Path):

        # create base KG
        await upsert_from_preprocessed(preprocess_chunks_filepath, preprocess_descriptions_filepath)

    async def pre_retrieve(self):
        # build LeanRAG KG with hierarchical clustering
        await build()
    
    async def retrieve(self, text, preprocess_chunks_filepath:Path):
        return await leanrag_retrieve.get_response_context_data(text, preprocess_chunks_filepath)
    
    async def reset(self):
        await mg_driver.clear()

/home/nathan/Projects/X-RAG/src/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from pathlib import Path
import json
import time
import dspy

MINE_DIRECTORY = Path("../datasets/MINE")


class EvalSignature(dspy.Signature):
	""" Does the context contain the information stated in the statement?. """
	context: str = dspy.InputField()
	statement: str = dspy.InputField()
	context_contains_statement: bool = dspy.OutputField()
eval = dspy.Predict(EvalSignature)

 
def score_count(result):
	""" Computes total score and count. """
	score = 0
	count = 0
	for part in result:
		for query in part["queries"]:
			count += 1
			score += int(query["contained"])
	return score, count


def conciseness(result):
	""" 
	For each "correct" response, how long is it? 
	
	Note: Having only one correct response that is concise will give a "good" score for this. 
	"""
	length = 0
	count = 0
	for part in result:
		for query in part["queries"]:
			if query["contained"]:
				length += len(query["context"])
				count += 1
	return length / count


def mean_median_query_time(result):
	times = []
	for part in result:
		for query in part["queries"]:
			times.append(query["duration"])
	mean = sum(times) / len(times)
	times.sort()
	median = times[len(times)//2]
	return mean, median


# Concurrency is not used here for now
# Maaaybe later 
# If I am annoyed enough! 
# TODO: timing is influenced by caching and I'm not sure what to do about it
from dataset_preprocess import process_dataset_file
JUDGE_MODEL = dspy.LM("bedrock/us.amazon.nova-pro-v1:0")

eval_results = []
eval_errors = []

async def miner_evaluate_individual_with_preprocess(miner: MINER, judge_model: str):
	result = []
	# long_essay = CWD / "../datasets/MINE/The Development of Renewable Energy Sources.json"
	# paths = [ long_essay ]
	paths = list(MINE_DIRECTORY.iterdir())
	for i, p in enumerate(paths): 
		try:
			print(f"Evaluate {p.name} ({i+1}/{len(paths)})")

			# Load data 
			with open(p, "r") as fp:
				mine_data = json.load(fp)

			# Creates pre-process data (chunks + g0 descriptions json files) - check if it exists already from prev runs
			preprocessed_chunks = CWD / f"{p.stem}__chunks.json"
			preprocessed_descs = CWD / f"{p.stem}__g0_descriptions.json"
			if not preprocessed_chunks.is_file() or preprocessed_descs.is_file():
				await process_dataset_file(p)

			# Ingest text 
			print("Ingest...")
			ingest_st = time.time()
			await miner.ingest(preprocessed_chunks, preprocessed_descs)
			await miner.pre_retrieve(p.name)
			ingest_en = time.time()
			# Query and evaluate 
			queries = []
			with dspy.context(lm=JUDGE_MODEL):
				for i, a in enumerate(mine_data["answers"]):
					print(f"\rQuery {i+1}/{len(mine_data["answers"])}", end="")
					q_st = time.time()
					info = await miner.retrieve(a, preprocessed_chunks)
					q_en = time.time()
					contained = (await eval.acall(context=info, statement=a)).context_contains_statement
					queries.append({
						"query": a,
						"context": info,
						"contained": contained,
						"duration": q_en - q_st,
					})
				print()
			result.append({
				"filename": p.name,
				"ingest_duration": ingest_en - ingest_st,
				"queries": queries,
			})
			await miner.reset()
		except Exception as e:
			result = {"error": str(e)}
	return result

In [6]:
import asyncio
from typing import Any
from aiolimiter import AsyncLimiter
from prettytable import PrettyTable

results_dir = Path("/tmp/miner")


async def evaluate(
	items: list[tuple[str, Any]],
	concurrency: int = 3,
):
	limiter = AsyncLimiter(concurrency)
	async def limited(f):
		async with limiter:
			return await f
	name_to_path = lambda n: results_dir / f"{n}.json"

	names, tasks = zip(*items)
	paths = [name_to_path(n) for n in names]

	print(f"Running {len(names)} evaluations with concurrency {concurrency}")
	if concurrency > 1:
		tasks = [limited(f) for f in tasks]
		results = await asyncio.gather(*tasks)
	else: 
		print("haha that's serial")
		results = [await f for f in tasks]
	print("Done!")

	results_dir.mkdir(exist_ok=True)
	for name, path, result in zip(names, paths, results):
		print(f"Saving '{path}'")
		with open(path, "w") as fp:
			json.dump({
				"name": name,
				"result": result,
			}, fp, indent=2)
	return paths


def show_results():
	errors = []
	table = PrettyTable()
	table.field_names = [
		"Name", 
		"Score", 
		"Context Length", 
		"Conciseness",
		"Query Duration (mean)", 
		"Query Duration (median)",
	]
	results_dir.mkdir(exist_ok=True)
	for f in results_dir.iterdir():
		with open(f, "r") as fp:
			data = json.load(fp)

		if ('error' in data['result']):
			errors.append(data['result']['error'])
		score, count = score_count(data["result"])
		r_conciseness = conciseness(data["result"])
		mean, median = mean_median_query_time(data["result"])
		table.add_row([
			data["name"], 
			f"{score/count*100:.2f}% ({score}/{count})", 
			f"{r_conciseness:.2f}",
			f"{score/count*100/r_conciseness:.2f}",
			f"{mean:.2f}s",
			f"{median:.2f}s",
		])
	print(table)
	print("ERORRS:")
	print(errors)

In [8]:
show_results()

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe7 in position 25: invalid continuation byte

In [ ]:
import upsert
import importlib
importlib.reload(upsert)
leanrag_miner = LeanragMINER()
result = await miner_evaluate_individual_with_preprocess(leanrag_miner, "bedrock/us.amazon.nova-pro-v1:0")
judge_model = ""
# await evaluate(
#     [("leanrag-default", miner_evaluate_individual_with_preprocess(leanrag_miner, judge_model))], concurrency = 1
# )